# Circuit Tracing Tutorial

<a target="_blank" href="https://colab.research.google.com/github/safety-research/circuit-tracer/blob/main/demos/circuit_tracing_tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook serves as a tutorial for the circuit tracing library. The library enables users to explain model behavior by generating Attribution Graphs (introduced in [Circuit Tracing](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)), and validate these explanations with experiments.

If you'd like to generate your own graphs, you can do so on [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-dallas-austin&pruningThreshold=0.6&pinnedIds=27_22605_10%2C20_15589_10%2CE_26865_9%2C21_5943_10%2C23_12237_10%2C20_15589_9%2C16_25_9%2C14_2268_9%2C18_8959_10%2C4_13154_9%2C7_6861_9%2C19_1445_10%2CE_2329_7%2CE_6037_4%2C0_13727_7%2C6_4012_7%2C17_7178_10%2C15_4494_4%2C6_4662_4%2C4_7671_4%2C3_13984_4%2C1_1000_4%2C19_7477_9%2C18_6101_10%2C16_4298_10%2C7_691_10&supernodes=%5B%5B%22capital%22%2C%2215_4494_4%22%2C%226_4662_4%22%2C%224_7671_4%22%2C%223_13984_4%22%2C%221_1000_4%22%5D%2C%5B%22state%22%2C%226_4012_7%22%2C%220_13727_7%22%5D%2C%5B%22Texas%22%2C%2220_15589_9%22%2C%2219_7477_9%22%2C%2216_25_9%22%2C%224_13154_9%22%2C%2214_2268_9%22%2C%227_6861_9%22%5D%2C%5B%22preposition+followed+by+place+name%22%2C%2219_1445_10%22%2C%2218_6101_10%22%5D%2C%5B%22capital+cities+%2F+say+a+capital+city%22%2C%2221_5943_10%22%2C%2217_7178_10%22%2C%227_691_10%22%2C%2216_4298_10%22%5D%5D&densityThreshold=0.99&clerps=%5B%5B%2223_2312237_10%22%2C%22Cities+and+states+names+%28say+Austin%29%22%5D%2C%5B%2218_1808959_10%22%2C%22state+%2F+regional+government%22%5D%5D). You can also do so directly on Colab using this [starter notebook](https://github.com/safety-research/circuit-tracer/blob/main/demos/attribute_demo.ipynb).

In this demo, we'll dive  into a couple of attribution graphs involving Gemma 2 (2B), based on the Multi-Step Reasoning and Multilingual Circuits examples [in the original paper](https://transformer-circuits.pub/2025/attribution-graphs/biology.html). We'll view the graphs, annotated with labeled supernodes, and then perform interventions to verify that the supernodes' labels are indeed correct, and that interventions have the effect we expect.

In [7]:
# @title Colab Setup Environment

try:
    import google.colab
    !curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

downloading uv 0.11.1 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.11.13 environment at: /usr
error: Distribution not found at: file:///content/circuit-tracer


In [2]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [10]:
import sys
# Reinstall transformers library
!{sys.executable} -m pip install --upgrade transformers
# Uninstall conflicting versions first to ensure a clean install
!pip uninstall -y transformers huggingface-hub hf-xet

# Install versions compatible with circuit-tracer
!pip install transformers==4.57.3 huggingface-hub==0.36.2

print("Transformers library reinstalled. Please restart the Colab runtime (Runtime > Restart runtime) and then re-run all cells.")

Transformers library reinstalled. Please restart the Colab runtime (Runtime > Restart runtime) and then re-run all cells.


In [6]:
from collections import namedtuple
from typing import List, Dict

import torch
import os

os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.demo_utils import extract_supernode_features

backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
model = ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend=backend)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [10]:
# Check what the model outputs for the prompt before tracing
prompt = "Answer with a Country Name. Which nation got the most Gold medals in the Olympics 2024?"

  # Use the built-in generation method (empty interventions = no changes)
output = model.feature_intervention_generate(prompt, [], do_sample=False, verbose=False)
print(output[0])

Answer with a Country Name. Which nation got the most Gold medals in the Olympics 2024?

Answer with a Country Name. Which nation got


In [11]:
graph = attribute(
    prompt=prompt,
    model=model,
    max_n_logits=10,
    desired_logit_prob=0.95,
    batch_size=256,
    max_feature_nodes=8192,
    offload="cpu",
    verbose=True,
)

Phase 0: Precomputing activations and vectors
Precomputation completed in 0.40s
Found 20983 active features
Phase 1: Running forward pass
Forward pass completed in 0.63s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.7539
Will include 8192 of 20983 feature nodes
Input vectors built in 1.81s
Phase 3: Computing logit attributions
Logit attributions completed in 0.71s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:14<00:00, 551.30it/s]
Feature attributions completed in 14.86s
Attribution completed in 22.19s


In [14]:
# Save graph
from pathlib import Path
graph_dir = Path("graphs")
graph_dir.mkdir(exist_ok=True)
graph.to_pt(graph_dir / "graph_try.pt")
print("Graph saved.")

Graph saved.


In [16]:
from circuit_tracer.utils.create_graph_files import create_graph_files
create_graph_files(
    graph_or_path=graph_dir / "graph_try.pt",
    slug="charlie-kirk-hallucination",
    output_path="./graph_files",
    node_threshold=0.8,
    edge_threshold=0.98,
)

In [22]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame, display
import google.colab.output
import socket

# Dynamically find a free port
def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

port = get_free_port()
server = serve(data_dir="./graph_files/", port=port)

# Generate a proxy URL to access the port on the Colab VM
proxy_url = google.colab.output.eval_js(f"google.colab.kernel.proxyPort({port})")
graph_url = f"{proxy_url}/index.html"

print(f"Open your graph at: {graph_url}")
display(IFrame(src=graph_url, width="100%", height="800px"))

Open your graph at: https://47715-gpu-l4-s-kkb-ass1a1-30lbgewdy7k2y-a.asia-southeast1-1.prod.colab.dev/index.html
